In [1]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))

# Connect
driver = GraphDatabase.driver(URI, auth=AUTH)

def run_query(query, params=None):
    """Helper to run a query and ignore the metadata"""
    if params is None:
        params = {}
    records, summary, _ = driver.execute_query(query, params, database_="neo4j")
    return records




In [2]:

def add_data():
    print("1. Creating Data...")
    
    # Create Students
    query_students = """
    UNWIND $students AS student
    MERGE (s:Student {name: student.name})
    SET s.major = student.major
    """
    students_data = [
        {"name": "Alice", "major": "CS"},
        {"name": "Bob", "major": "Math"},
        {"name": "Charlie", "major": "CS"}
    ]
    run_query(query_students, {"students": students_data})

    # Create Courses
    query_courses = """
    UNWIND $courses AS course
    MERGE (c:Course {name: course})
    """
    courses_data = ["Graph Theory 101", "Linear Algebra", "Python Programming"]
    run_query(query_courses, {"courses": courses_data})
    
    print("   -> Students and Courses added.")

add_data()

1. Creating Data...
   -> Students and Courses added.


In [3]:
def enroll_students():
    print("2. Enrolling Students...")
    
    # Alice takes Python and Graph Theory
    # Bob takes Linear Algebra
    # Charlie takes Python
    
    query = """
    MATCH (s:Student {name: $student_name})
    MATCH (c:Course {name: $course_name})
    MERGE (s)-[:ENROLLED_IN {semester: 'Spring 2026'}]->(c)
    """
    
    enrollments = [
        ("Alice", "Python Programming"),
        ("Alice", "Graph Theory 101"),
        ("Bob", "Linear Algebra"),
        ("Charlie", "Python Programming")
    ]
    
    for student, course in enrollments:
        run_query(query, {"student_name": student, "course_name": course})
    
    print("   -> Relationships created.")

enroll_students()

2. Enrolling Students...
   -> Relationships created.


In [ ]:
def find_classmates(course_name):
    print(f"\n3. Who is taking {course_name}?")
    
    query = """
    MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
    WHERE c.name = $course_name
    RETURN s.name, s.major
    """
    
    results = run_query(query, {"course_name": course_name})
    
    for record in results:
        print(f"   - Found Student: {record['s.name']} ({record['s.major']})")

find_classmates("Python Programming")

In [4]:
def suggest_friends(student_name):
    print(f"\n4. Friend Recommendations for {student_name}:")
    
    # "Find other students (other) who are enrolled in the same course (c) as our student (s)"
    query = """
    MATCH (s:Student {name: $name})-[:ENROLLED_IN]->(c:Course)<-[:ENROLLED_IN]-(other:Student)
    RETURN other.name AS recommended_friend, c.name AS common_interest
    """
    
    results = run_query(query, {"name": student_name})
    
    if not results:
        print("   No recommendations found based on shared classes.")
    else:
        for record in results:
            print(f"   - You should meet {record['recommended_friend']} (Common Class: {record['common_interest']})")

suggest_friends("Alice")


4. Friend Recommendations for Alice:
   - You should meet Charlie (Common Class: Python Programming)
